# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lthendral10/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
!git clone https://github.com/lthendral10/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [11]:
!find /content -name "content_refresh_anonymized.csv"

/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [12]:
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Build the Feature Vector

The feature vector contains SEO, engagement, and content-related variables that are available before making a prediction. Missing numeric values are filled, and categorical features are encoded so they can be used in machine learning models.

In [14]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Selected features
feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

X = df[feature_columns].copy()

# Fill missing numeric values
X = X.fillna(X.median(numeric_only=True))

print("Feature Vector Shape:", X.shape)
X.head()

Feature Vector Shape: (30000, 14)


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,17,187,20,0.76,10.6,5.88,4.55
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,9,445,25,0.05,20.3,0.00,10.00
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,11,141,20,0.09,36.5,0.00,28.57
3,10.0,0.00,0.00,2877.0,19116.0,11751,58,78,463,22,0.49,6.2,1.28,3.45
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,145,263,14,0.13,44.0,0.00,24.29


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


| Feature                | Meaning                        | Missing Value Handling | Available Before Prediction? |
| ---------------------- | ------------------------------ | ---------------------- | ---------------------------- |
| search_volume          | Monthly search demand          | Filled with median     | Yes                          |
| ctr                    | Click-through rate             | Filled with median     | Yes                          |
| avg_position           | Average search ranking         | Filled with median     | Yes                          |
| days_since_last_update | Days since content was updated | Filled with median     | Yes                          |
| engagement_rate        | User engagement                | Filled with median     | Yes                          |
| scroll_rate            | Scroll percentage              | Filled with median     | Yes                          |


In [16]:
feature_notes = pd.DataFrame({
    "Feature": feature_columns,
    "Missing Values": [df[col].isnull().sum() for col in feature_columns],
    "Data Type": [df[col].dtype for col in feature_columns]
})

feature_notes

,Feature,Missing Values,Data Type
0,search_volume,2468,float64
1,competition,2468,float64
2,cpc,2468,float64
3,word_count,7699,float64
4,char_count,7699,float64
5,impressions_90d,0,int64
6,clicks_90d,0,int64
7,sessions_90d,0,int64
8,content_age_days,0,int64
9,days_since_last_update,0,int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Leakage Hunt

I checked every feature to ensure that it does not reveal future information or directly encode the prediction target.

I excluded identifiers and avoided features that could contain future outcomes or private information.

In [18]:
# Features not used because they may leak information
possible_leakage = [
    "content_id",
    "client_id"
]

print("Potential Leakage Columns:")
print(possible_leakage)

print("\nChecking if leakage columns exist:")
for col in possible_leakage:
    print(col, "->", col in df.columns)

Potential Leakage Columns:
['content_id', 'client_id']

Checking if leakage columns exist:
content_id -> True
client_id -> True


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


| Excluded Field | Reason                                                                                           |
| -------------- | ------------------------------------------------------------------------------------------------ |
| content_id     | Identifier only; not predictive.                                                                 |
| client_id      | Client identifier; not useful for general prediction and may encode client-specific information. |
| provider_used  | Operational metadata; not directly related to content performance.                               |
| model_used     | Operational metadata; not a content signal.                                                      |


In [20]:
# Columns excluded from the feature vector
excluded_features = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used"
]

# If trend_direction is used as the target, exclude it from features
excluded_features.append("trend_direction")

print("Excluded Features:")
for feature in excluded_features:
    print("-", feature)

print("\nNumber of excluded features:", len(excluded_features))

# Verify that these columns exist in the dataset
print("\nExcluded columns present in dataset:")
print(df[excluded_features].head())

Excluded Features:
- content_id
- client_id
- provider_used
- model_used
- trend_direction

Number of excluded features: 5

Excluded columns present in dataset:
             content_id          client_id provider_used  \
0  content_304f48230142  client_f369cb89fc           NaN   
1  content_a1fb4e703a9e  client_4e07408562           NaN   
2  content_9aa793d4d895  client_7f2253d7e2           NaN   
3  content_331d6c4de07b  client_19581e27de           NaN   
4  content_d99b7a2d90ca  client_3fdba35f04           NaN   

               model_used trend_direction  
0        gemini-2.5-flash            down  
1  gemini-3-flash-preview            down  
2        gemini-2.5-flash            down  
3                     NaN          stable  
4  gemini-3-flash-preview            down  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.